# Homework — Fitting a line to data (linear regression)

A huge amount of chemistry comes down to one question: **as I change X, how does Y respond?**
Concentration vs. absorbance, temperature vs. rate, dose vs. signal. When the relationship
looks like a straight line, we *fit a line* to summarize it in two numbers — a **slope** and
an **intercept** — and then use those numbers to interpret and predict.

**How this works**
1. Run the setup cell below (installs the tools and imports them).
2. Read each short teaching section and run its example — the expected output is shown as a `# comment`.
3. Answer the numbered **Problems**. Some ask for code; several ask you to *write a sentence or two of reasoning* — those matter just as much.
4. At the very end there's a **check-in value** — type it into the CHEM 438 app to mark this homework done.

Run every cell top to bottom. The whole point of this homework is **interpretation**, not just getting a number to print.

In [ ]:
# Setup — run this first
!pip install numpy scipy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
print("Ready to fit.")

## 1. The idea: a line that "best" goes through the data

Real measurements are never perfectly on a line — there's always a bit of noise. Fitting a
line means finding the *one straight line* that comes closest to all the points at once. That
line is described by the familiar equation

$$y = (\text{slope}) \cdot x + (\text{intercept})$$

The slope tells you **how much y changes for each 1-unit increase in x**, and the intercept
tells you **what y would be when x = 0**. Below is a small dataset of a UV-Vis experiment:
the concentration of a dye (mg/mL) versus its measured absorbance. Look at how the points
*almost* fall on a line.

In [ ]:
concentration = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])   # dye concentration, mg/mL
absorbance    = np.array([0.04, 0.20, 0.33, 0.51, 0.62, 0.79])  # measured absorbance (unitless)

plt.scatter(concentration, absorbance)
plt.xlabel("concentration (mg/mL)")
plt.ylabel("absorbance")
plt.title("The points rise almost in a straight line")
plt.show()
# You should see 6 dots trending up and to the right, nearly (but not exactly) on a line.

## 2. Fitting the line with `scipy.stats.linregress`

You don't eyeball the line — you compute it. `linregress(x, y)` does all the math and hands
back a result object. The three pieces you'll use constantly:

- `result.slope` — the slope
- `result.intercept` — the intercept
- `result.rvalue` — the correlation *r*; square it to get **R²** (next section)

In [ ]:
result = linregress(concentration, absorbance)

print("slope     =", round(result.slope, 3))       # 0.148
print("intercept =", round(result.intercept, 3))   # 0.044
print("r value   =", round(result.rvalue, 3))      # 0.999

## 3. What the slope and intercept actually MEAN

Numbers are useless until you attach units and a sentence to them. For this fit:

- **Slope = 0.148 per (mg/mL).** Because absorbance is unitless and concentration is in mg/mL,
  the slope means: *every extra 1 mg/mL of dye adds about 0.148 to the absorbance.* This is
  literally the sensitivity of the measurement — a steeper slope means a more responsive assay.
- **Intercept = 0.044.** This is the predicted absorbance at *zero* concentration. Ideally it
  would be 0, but instruments have a small baseline (the cuvette, the solvent). A nonzero
  intercept here is a **blank / baseline signal**, not dye.

Always read a fit back as a sentence in real units. "The slope is 0.148" is half an answer;
"absorbance rises 0.148 for every 1 mg/mL of dye" is the whole answer.

In [ ]:
slope = result.slope
intercept = result.intercept
print(f"Each +1 mg/mL of dye raises absorbance by {slope:.3f}")   # 0.148
print(f"Baseline absorbance at 0 mg/mL is about {intercept:.3f}")  # 0.044

## 4. R² — how much of the variation the line explains

**R²** ranges from 0 to 1 and answers: *of all the up-and-down variation in y, what fraction
does the line account for?*

- **R² = 1.0** → the line explains 100% of the variation; every point sits exactly on it.
- **R² = 0.90** → the line explains 90% of the variation; the remaining 10% is scatter the
  line can't capture (noise, or a missing effect).
- **R² = 0.0** → the line explains nothing; knowing x tells you nothing about y.

Get R² by squaring `rvalue`. Don't just print it — *say what it means.*

In [ ]:
r_squared = result.rvalue ** 2
print("R^2 =", round(r_squared, 3))   # 0.998
# Interpretation: the line explains about 99.8% of the variation in absorbance.
# Only ~0.2% is leftover scatter. This is an excellent straight-line relationship.

## 5. Making a prediction from the fit

Once you have slope and intercept, the line becomes a *prediction machine*:
`y_predicted = slope * x + intercept`. Suppose you measure a new sample and want the
absorbance you'd expect at 2.5 mg/mL:

In [ ]:
x_new = 2.5
y_pred = slope * x_new + intercept
print(f"Predicted absorbance at {x_new} mg/mL = {y_pred:.3f}")   # 0.415

# One caution: 2.5 is INSIDE our measured range (0 to 5), so this is *interpolation* — trustworthy.
# Predicting at x = 50 would be *extrapolation* — far outside the data, where the line may not hold.

## 6. Plotting the data and the fit line together

Always plot the fit *on top of* the data. The line is your model; the dots are reality.
Seeing them together is how you judge whether the fit is any good — a number like R² can
hide problems a picture reveals (you'll see this in the next section).

In [ ]:
plt.scatter(concentration, absorbance, label="data")

# build the fit line across the x-range
x_line = np.linspace(concentration.min(), concentration.max(), 100)
y_line = slope * x_line + intercept
plt.plot(x_line, y_line, color="red", label=f"fit: y = {slope:.3f}x + {intercept:.3f}")

plt.xlabel("concentration (mg/mL)")
plt.ylabel("absorbance")
plt.legend()
plt.title(f"Fit explains R^2 = {r_squared:.3f} of the variation")
plt.show()
# The red line threads cleanly through all six dots — a trustworthy fit.

## 7. When a straight line is the WRONG model

A line is only right when the true relationship *is* a line. Plenty of chemistry isn't:
reactions saturate, binding curves plateau, signals level off. Here is a saturating dataset —
it climbs fast, then flattens. Watch what happens when we force a line through it.

In [ ]:
x_sat = np.array([1.,2.,3.,4.,5.,6.,7.,8.])
y_sat = np.array([2.0, 3.4, 4.3, 5.0, 5.5, 5.8, 6.0, 6.1])  # rises then plateaus

fit_sat = linregress(x_sat, y_sat)
xl = np.linspace(1, 8, 100)
plt.scatter(x_sat, y_sat, label="data (curves!)")
plt.plot(xl, fit_sat.slope*xl + fit_sat.intercept, color="red", label="straight-line fit")
plt.legend(); plt.xlabel("x"); plt.ylabel("y")
plt.title(f"R^2 = {fit_sat.rvalue**2:.3f}, but look at the shape")
plt.show()

print("R^2 =", round(fit_sat.rvalue**2, 3))   # 0.883

**Read that plot carefully.** The R² is 0.883 — not terrible! If you only printed the
number you might call it a good fit. But the *picture* tells a different story: the line sits
**below** the data in the middle and **above** it at both ends. The points don't scatter
*randomly* around the line — they curve away from it in a systematic pattern. That structured
miss is the tell-tale sign that **the model is wrong, not just noisy**. When residuals bend
the same way instead of scattering evenly, a line is the wrong tool — you'd reach for a curved
model (a log, a saturation/Michaelis-Menten equation, a polynomial) instead.

*(Aside: `scipy.stats.linregress` is the simplest fitting tool and all you need here. You may
later meet `sklearn.linear_model.LinearRegression`, which does the same straight-line fit with
a heavier, more general interface — same idea, more machinery.)*

## Problems

Write your answer in the empty cell under each problem. **Code problems** need a code cell;
**reasoning problems** ask you to type a few sentences in the answer cell provided. The
reasoning ones are graded on whether your *interpretation* is right, not on any number.

**Problem 1 (compute).** A kinetics experiment gives these paired measurements:

```
x = [1., 2., 3., 4., 5., 6.]
y = [2.3, 4.1, 6.0, 7.8, 10.2, 11.9]
```

Fit a line with `linregress` and print the **slope**, **intercept**, and **R²** (round each
to 3 decimals).

**Problem 2 (reasoning — interpret the slope).** A calibration fit for a protein assay comes
out as **slope = 2.5, intercept = 1.2**, where x is protein mass in **mg** and y is signal
volume in **mL**.

In the answer cell below, write 2–3 sentences answering: *What does the slope of 2.5 mean
physically, in real units? What does the intercept of 1.2 represent?* Be specific about units
(mL per mg) — don't just restate the numbers.

*Your answer (double-click to edit this cell and type here):*

...

**Problem 3 (reasoning — interpret R²).** You fit two different calibration curves. Fit A has
**R² = 0.95**; Fit B has **R² = 0.40**.

In the answer cell, write 2–3 sentences: *Which fit would you trust more to make predictions,
and why? What does the number 0.95 literally mean about the variation in the data?* (Hint: R²
is the fraction of variation the line explains.)

*Your answer:*

...

**Problem 4 (reasoning — prediction & its limits).** A fit comes out as **y = 3x + 1**, and it
was built from data where x ranged from **0 to 8**.

In the answer cell: (a) what does the fit predict at **x = 10**? (show the arithmetic), and
(b) is predicting at x = 10 *interpolation* or *extrapolation*, and why does the distinction
matter for how much you trust the answer?

*Your answer:*

...

**Problem 5 (compute — make a prediction).** Here is a standard curve:

```
x = [0., 10., 20., 30., 40.]
y = [1.0, 3.1, 5.0, 6.9, 9.1]
```

Fit a line, then use the slope and intercept to **predict y at x = 25**. Print the prediction
rounded to 2 decimals. (x = 25 is inside the measured range, so this is interpolation.)

**Problem 6 (reasoning — critique the fit).** Reread section 7. A classmate fits a line to a
dataset, gets **R² = 0.88**, and declares "great fit!" But the scatter plot shows the points
bending in a smooth curve, sitting below the line in the middle and above it at the ends.

In the answer cell: *What specifically tells you the line is the wrong model here — and why is
looking at the plot necessary even though R² is fairly high?* Mention both the R² and what the
plot shows.

*Your answer:*

...

**Problem 7 (compare).** Two datasets share the same x values:

```
x     = [1., 2., 3., 4., 5.]
tight = [2.1, 3.9, 6.1, 7.9, 10.0]
noisy = [3.5, 2.5, 4.5, 9.5, 10.0]
```

Fit a line to **each**. You'll find they have *almost the same slope* (~2) but very different
R². Print both slopes and both R² values, then in a comment answer: *the slopes are nearly
identical, so what is actually different between these two datasets?*

**Problem 8 (concept check — multiple choice).** A fit gives **R² = 0.30**. Which statement is
the best interpretation?

- **(A)** The slope is 0.30.
- **(B)** The line explains about 30% of the variation in y; 70% is scatter the line doesn't capture.
- **(C)** 30% of the data points lie exactly on the line.
- **(D)** The prediction at x = 30 will be off by 30%.

In the answer cell, write the single correct letter and one sentence saying why the others are wrong.

*Your answer:*

...

## Check in

Run the cell below. It builds a **seeded** dataset (so everyone gets the identical result),
fits a line, and prints one number — the **slope, rounded to 2 decimals**.
**Type that number into the CHEM 438 app to mark this homework complete.**

In [ ]:
np.random.seed(438)
x = np.linspace(0, 10, 25)
y = 1.8 * x + 5 + np.random.normal(0, 2, size=25)   # a noisy straight-line relationship

fit = linregress(x, y)
print("slope =", round(fit.slope, 2))
print("R^2   =", round(fit.rvalue**2, 2))   # (for your interest: how much the line explains)
print()
print("Your check-in value (the slope):", round(fit.slope, 2))